# UE boss damage simulation
This attempts to simulate hitting bosses with certain lt configurations.

For now it supports Overkill, Tanya, The collector, Simba and leopard. 

It doesn't consider lt crits since we don't know the proc chance for that and we assume a baseline of crits dealing double damage without any further investment into crit dmg.

In [102]:
BASE_CRIT_DAMAGE = 100 # this means crit deal double dmg by default

In [103]:
import random
import pandas as pd
from typing import List, Tuple, Dict

def simulate_damage(
    base_damage: float = 10.0,
    percent_damage_mod: float = 0.0,
    percent_boss_damage_mod: float = 0.0,
    crit_chance: float = 25.0,
    crit_damage: float = 50.0,
    num_hits: int = 10000,
    use_ok: bool = False,
    ok_proc_chance: float = 10.0,
    use_collector: bool = True,
    collector_multiplier: float = 3.0,
    use_tanya: bool = False,
    tanya_proc_chance: float = 30,
    tanya_dmg: int = 50,
    use_caio: bool = False,
    caio_proc_chance: float = 15.0,
    use_leopard: bool = False,
    leopard_multiplier: float = 2.0,
    use_simba: bool = False,
    simba_proc_chance: float = 4.0,
    use_silas: bool = False,
    silas_proc_chance: float = 5.0,
) -> Tuple[List[float], List[Dict[str, float]]]:
    """
    Simulate damage for a series of hits, with optional critical hits,
    overkill procs, Tanya procs, collector procs, Caio procs, simba procs and leopard procs.
    """
    hits = []
    proc_data = []

    for _ in range(num_hits):
        
        silas_proc_occurred = False
        is_crit = False
        collector_proc_occurred = False
        tanya_proc_occurred = False
        ok_proc_occurred = False
        ok_proc_depth = 0
        leopard_proc_occured = False
        simba_proc_occured = False
        
        if use_silas and (random.random() < silas_proc_chance / 100):
            modified_base_damage = base_damage * 2
            silas_proc_occurred = True
        else:
            modified_base_damage = base_damage
            
        # Calculate the regular damage with generic damage and boss damage modifiers
        regular_damage = modified_base_damage * (1 + percent_damage_mod / 100) * (1 + percent_boss_damage_mod / 100)

        
        # Determine if the hit is a critical hit
        if random.random() < crit_chance / 100:
            is_crit = True
            crit_extra_damage = regular_damage * ((crit_damage + BASE_CRIT_DAMAGE) / 100)
            damage = regular_damage + crit_extra_damage
            
            if use_ok:
                # Handle Overkill chain calculation
                multiplier = 1
                while random.random() < ok_proc_chance / 100:
                    ok_proc_occurred = True
                    ok_proc_depth += 1
                    multiplier *= 2
                    if multiplier >= 64:
                        break
                crit_extra_damage *= multiplier
                damage = regular_damage + crit_extra_damage

            if use_tanya:
                if random.random() < tanya_proc_chance / 100:
                    tanya_proc_occurred = True
                    damage *= 1 + tanya_dmg / 100

            # Handle the collector proc (only on crit)
            if use_collector:
                if random.random() < 15 / 100:
                    collector_proc_occurred = True
                    damage *= collector_multiplier
        else:
            damage = regular_damage
            
        if use_caio:
            if random.random() < caio_proc_chance / 100:
                caio_proc_occurred = True
                damage *= 2

        # handle sk procs
        if use_leopard:
            if random.random() < 10 / 100:
                damage *= leopard_multiplier
                leopard_proc_occured = True

        if use_simba:
            if not leopard_proc_occured:
                 if random.random() < simba_proc_chance / 100:
                    damage *= 3
                    simba_proc_occured = True

        hits.append(damage)
        # Build the proc_data dictionary conditionally
        proc_entry = {
            'Damage': damage,
            'IsCrit': is_crit,
        }

        if use_collector:
            proc_entry['CollectorProcOccurred'] = collector_proc_occurred
        
        if use_ok:
            proc_entry['OKProcOccurred'] = ok_proc_occurred
            proc_entry['OKProcDepth'] = ok_proc_depth
        
        if use_tanya:
            proc_entry['TanyaProcOccurred'] = tanya_proc_occurred
            
        if use_caio:
            proc_entry['CaioProcOccurred'] = caio_proc_occurred

        if use_leopard:
            proc_entry['LeopardProcOccurred'] = leopard_proc_occured

        if use_simba:
            proc_entry['SimbaProcOccurred'] = simba_proc_occured
            
        if use_silas:
            proc_entry['SilasProcOccurred'] = silas_proc_occurred

        proc_data.append(proc_entry)

    return hits, proc_data

## MMB simulation, 100% crit chance with Tanya and Collector

In [104]:
#Example usage
num_hits = 100000
base_damage = 10
crit_chance = 100
crit_damage = 291
use_ok = False
ok_proc_chance = 24
percent_damage_mod = 179
use_collector = True
collector_multiplier = 3.5  # Multiplier for the entire hit
use_tanya = False
tanya_proc_chance = 30
tanya_dmg = 140
use_silas = True
silas_proc_chance = 19

# num_hits = 100000
# base_damage = 10
# crit_chance = 100
# crit_damage = 156
# use_ok = False
# ok_proc_chance = 24
# percent_damage_mod = 112
# use_collector = True
# collector_multiplier = 3.5  # Multiplier for the entire hit
# use_tanya = True
# tanya_proc_chance = 30
# tanya_dmg = 110

# Initialize lists to store results from each run
all_hits = []
all_proc_data = []

hits, proc_data = simulate_damage(
    base_damage=base_damage, 
    percent_damage_mod=percent_damage_mod, 
    crit_chance=crit_chance, 
    crit_damage=crit_damage, 
    num_hits=num_hits, 
    use_ok=use_ok, 
    use_collector=use_collector,
    ok_proc_chance=ok_proc_chance, 
    collector_multiplier=collector_multiplier,
    use_tanya=use_tanya,
    tanya_proc_chance=tanya_proc_chance,
    tanya_dmg=tanya_dmg,
    use_silas=use_silas,
    silas_proc_chance=silas_proc_chance)

# Convert hits and proc data to a pandas DataFrame for analysis
df_hits = pd.DataFrame(hits, columns=['Damage'])
df_proc_data = pd.DataFrame(proc_data)

# Basic statistics
print(df_hits.describe())
print("\n")
print(df_proc_data.value_counts())

              Damage
count  100000.000000
mean      223.931809
std       169.224628
min       136.989000
25%       136.989000
50%       136.989000
75%       273.978000
max       958.923000


Damage    IsCrit  CollectorProcOccurred  SilasProcOccurred
136.9890  True    False                  False                68796
273.9780  True    False                  True                 16196
479.4615  True    True                   False                12222
958.9230  True    True                   True                  2786
Name: count, dtype: int64


## Jared, tanya, yugo, collector, slaughterer

In [105]:
# Example usage
num_hits = 100000
base_damage = 10
crit_chance = 100
crit_damage = 600
use_ok = True
ok_proc_chance = 22
percent_damage_mod = 50
use_collector = True
collector_multiplier = 3.5  # Multiplier for the entire hit
use_tanya = True
tanya_proc_chance = 30
tanya_dmg = 110
use_caio = False

# Initialize lists to store results from each run
all_hits = []
all_proc_data = []

hits, proc_data = simulate_damage(
    base_damage=base_damage, 
    percent_damage_mod=percent_damage_mod, 
    crit_chance=crit_chance, 
    crit_damage=crit_damage, 
    num_hits=num_hits, 
    use_ok=use_ok, 
    use_collector=use_collector,
    ok_proc_chance=ok_proc_chance, 
    collector_multiplier=collector_multiplier,
    use_tanya=use_tanya,
    tanya_proc_chance=tanya_proc_chance,
    tanya_dmg=tanya_dmg,
    use_caio=use_caio)

# Convert hits and proc data to a pandas DataFrame for analysis
df_hits = pd.DataFrame(hits, columns=['Damage'])
df_proc_data = pd.DataFrame(proc_data)

# Basic statistics
print(df_hits.describe())
print("\n")
print(df_proc_data.value_counts())

              Damage
count  100000.000000
mean      295.373722
std       438.429579
min       120.000000
25%       120.000000
50%       225.000000
75%       252.000000
max     49502.250000


Damage    IsCrit  CollectorProcOccurred  OKProcOccurred  OKProcDepth  TanyaProcOccurred
120.00    True    False                  False           0            False                46220
252.00    True    False                  False           0            True                 20035
225.00    True    False                  True            1            False                10270
420.00    True    True                   False           0            False                 8162
472.50    True    False                  True            1            True                  4403
882.00    True    True                   False           0            True                  3461
435.00    True    False                  True            2            False                 2190
787.50    True    True                   T

## MMB simulation, 100% crit chance 
Tanya, Yugo, Collector, Jared, Slaughterer

In [106]:
# Example usage
num_hits = 100000
base_damage = 10
crit_chance = 100
crit_damage = 279.24
use_ok = False
ok_proc_chance = 24
percent_damage_mod = 132.25
use_collector = True
collector_multiplier = 3.5  # Multiplier for the entire hit
use_tanya = True
tanya_proc_chance = 30
tanya_dmg = 90

# Initialize lists to store results from each run
all_hits = []
all_proc_data = []

hits, proc_data = simulate_damage(
    base_damage=base_damage, 
    percent_damage_mod=percent_damage_mod, 
    crit_chance=crit_chance, 
    crit_damage=crit_damage, 
    num_hits=num_hits, 
    use_ok=use_ok, 
    use_collector=use_collector,
    ok_proc_chance=ok_proc_chance, 
    collector_multiplier=collector_multiplier,
    use_tanya=use_tanya,
    tanya_proc_chance=tanya_proc_chance,
    tanya_dmg=tanya_dmg)

# Convert hits and proc data to a pandas DataFrame for analysis
df_hits = pd.DataFrame(hits, columns=['Damage'])
df_proc_data = pd.DataFrame(proc_data)

# Basic statistics
print(df_hits.describe())
print("\n")
print(df_proc_data.value_counts())

              Damage
count  100000.000000
mean      194.822401
std       147.695700
min       111.303490
25%       111.303490
50%       111.303490
75%       211.476631
max       740.168208


Damage      IsCrit  CollectorProcOccurred  TanyaProcOccurred
111.303490  True    False                  False                59357
211.476631  True    False                  True                 25647
389.562215  True    True                   False                10404
740.168208  True    True                   True                  4592
Name: count, dtype: int64


## MMB simulation, 100% crit chance with Yugo and Collector

In [107]:
# Example usage
num_hits = 100000
base_damage = 10
crit_chance = 100
crit_damage = 279
use_ok = False
ok_proc_chance = 24
percent_damage_mod = 122
collector_multiplier = 3.5  # Multiplier for the entire hit
use_tanya = False
tanya_dmg = 110

# Initialize lists to store results from each run
all_hits = []
all_proc_data = []

hits, proc_data = simulate_damage(
    base_damage=base_damage, 
    percent_damage_mod=percent_damage_mod, 
    crit_chance=crit_chance, 
    crit_damage=crit_damage, 
    num_hits=num_hits, 
    use_ok=use_ok, 
    ok_proc_chance=ok_proc_chance, 
    collector_multiplier=collector_multiplier,
    use_tanya=use_tanya,
    tanya_dmg=tanya_dmg)

# Convert hits and proc data to a pandas DataFrame for analysis
df_hits = pd.DataFrame(hits, columns=['Damage'])
df_proc_data = pd.DataFrame(proc_data)

# Basic statistics
print(df_hits.describe())
print("\n")
print(df_proc_data.value_counts())

              Damage
count  100000.000000
mean      145.813324
std        94.530977
min       106.338000
25%       106.338000
50%       106.338000
75%       106.338000
max       372.183000


Damage   IsCrit  CollectorProcOccurred
106.338  True    False                    85151
372.183  True    True                     14849
Name: count, dtype: int64


In [108]:
# Example usage
num_hits = 100000
base_damage = 10
crit_chance = 47
crit_damage = 152
percent_damage_mod = 98
collector_multiplier = 3.5  # Multiplier for the entire hit

# Initialize lists to store results from each run
all_hits = []
all_proc_data = []

hits, proc_data = simulate_damage(
    base_damage=base_damage, 
    percent_damage_mod=percent_damage_mod, 
    crit_chance=crit_chance, 
    crit_damage=crit_damage, 
    num_hits=num_hits, 
    use_ok=use_ok, 
    ok_proc_chance=ok_proc_chance, 
    collector_multiplier=collector_multiplier,
    use_tanya=use_tanya,
    tanya_dmg=tanya_dmg)

# Convert hits and proc data to a pandas DataFrame for analysis
df_hits = pd.DataFrame(hits, columns=['Damage'])
df_proc_data = pd.DataFrame(proc_data)

# Basic statistics
print(df_hits.describe().round(2))
print("\n")
print(df_proc_data.value_counts())

          Damage
count  100000.00
mean       55.29
std        56.89
min        19.80
25%        19.80
50%        19.80
75%        69.70
max       243.94


Damage   IsCrit  CollectorProcOccurred
19.800   False   False                    53233
69.696   True    False                    39790
243.936  True    True                      6977
Name: count, dtype: int64
